# 01 · Explore the study lakes

A first look at the project's 9 lake outlines and the Sentinel-2 imagery we will use.
Copy this notebook as a starting point for your own.

**How to use it in Colab**
1. Run the cells from top to bottom (**Runtime → Run all**).
2. When asked, sign in with your **UoN account** (`@students.uonbi.ac.ke`).
3. To save changes back to GitHub: **Edit → Clear all outputs**, then **File → Save a copy in GitHub**, and pick **your own branch** (never `main`).


## 1. Setup: get the repo (run first in every session)

In [ ]:
import os
REPO = "/content/kenya-lakes-monitor"

if not os.path.exists(REPO):                       # first run this session: download the repo
    !git clone -q https://github.com/Giftmasil/kenya-lakes-monitor.git {REPO}
%cd {REPO}
!git pull -q                                       # latest version of main
!pip install -q -r requirements.txt                # install what Colab doesn't already have

## 2. The 9 study lakes

In [ ]:
import geopandas as gpd

lakes = gpd.read_file("data/lakes/lakes.geojson")
lakes[["lake_id", "name", "type", "split", "hydrolakes_area_km2", "wofs_max_area_km2", "aoi_area_km2"]]

Interactive map of the outlines, coloured by lake type. Zoom in and click a lake for its details.

The outlines are **search areas** (the largest water extent ever seen, plus a 1 km margin), not the lakes' sizes.

In [ ]:
lakes.explore(column="type", tooltip=["name", "type", "split", "aoi_area_km2"], tiles="Esri.WorldImagery")

## 3. Connect to Earth Engine

In [ ]:
import ee

ee.Authenticate()                                  # pick your UoN account in the pop-up
ee.Initialize(project="kenya-lakes-monitor")
print("Earth Engine ready")

## 4. How many Sentinel-2 images does each lake have?

Counts the images in the January–February dry season of a given year (the season the project uses).

In [ ]:
YEAR = 2025

s2 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED").filterDate(f"{YEAR}-01-01", f"{YEAR}-03-01")

counts = {}
for _, lake in lakes.iterrows():
    geom = ee.Geometry(lake.geometry.__geo_interface__)
    counts[lake.lake_id] = s2.filterBounds(geom).size().getInfo()

counts

## 5. A cloud-free true-colour picture of one lake

Builds a median composite of the dry-season images, with clouds removed using Cloud Score+
(the same method Melissa's pipeline will use). Change `LAKE` to see another one
(`baringo`, `bogoria`, `solai`, `nakuru`, `elementaita`, `naivasha`, `ol_bolossat`, `magadi`, `turkana`).

In [ ]:
from IPython.display import Image, display

LAKE = "naivasha"
CLEAR_THRESHOLD = 0.60                             # Cloud Score+: keep pixels at least this clear

lake = lakes[lakes.lake_id == LAKE].iloc[0]
geom = ee.Geometry(lake.geometry.__geo_interface__)

cloud_score = ee.ImageCollection("GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED")
composite = (s2.filterBounds(geom)
               .linkCollection(cloud_score, ["cs_cdf"])
               .map(lambda img: img.updateMask(img.select("cs_cdf").gte(CLEAR_THRESHOLD)))
               .median()
               .clip(geom))

url = composite.getThumbURL({"bands": ["B4", "B3", "B2"], "min": 0, "max": 2500,
                             "region": geom, "dimensions": 768, "format": "png"})
print(f"{lake['name']}, Jan–Feb {YEAR}")
display(Image(url=url))

## 6. Same lake, water highlighted (MNDWI)

MNDWI = (Green − SWIR1) / (Green + SWIR1). Water is bright (above 0), land is dark.
This simple index is the **baseline** our U-Net model has to beat.

In [ ]:
mndwi = composite.normalizedDifference(["B3", "B11"]).rename("mndwi")

url = mndwi.getThumbURL({"min": -0.6, "max": 0.6, "palette": ["8c510a", "f6e8c3", "c7eae5", "01665e"],
                         "region": geom, "dimensions": 768, "format": "png"})
display(Image(url=url))

water_km2 = (mndwi.gt(0).selfMask().multiply(ee.Image.pixelArea())
                .reduceRegion(ee.Reducer.sum(), geom, scale=20, maxPixels=1e10)
                .getInfo()["mndwi"] / 1e6)
print(f"Rough open-water area from MNDWI > 0: {water_km2:.1f} km²  (outline area: {lake.aoi_area_km2} km²)")

## Next steps

- **Aneselmus:** yearly lake areas from WOfS (plan tasks A1–A3).
- **Melissa:** export these composites + JRC water labels as training tiles (M1–M3).
- Put reusable code in `src/`, keep notebooks for exploring.
